# 🤖 Bot RAG que responde preguntas sobre Ventas

Este notebook implementa un sistema de **Recuperación Aumentada con Generación (RAG)** usando:
- 📄 **Corpus**: archivo `ventas.csv` como base de conocimiento
- 🔍 **RAG**: recuperación de contexto relevante con embeddings
- 🧠 **Mistral**: modelo de lenguaje para generar respuestas
- 🤝 **Agentes**: pipeline inteligente de pregunta → búsqueda → respuesta

---

## 📦 Celda 1: Instalación de dependencias

In [10]:
!pip install -q transformers sentence-transformers faiss-cpu pandas mistralai langchain langchain-community

## 📁 Celda 2: Cargar el archivo ventas.csv

> Sube tu archivo `ventas.csv` a Colab o ejecuta la celda siguiente para generar uno de ejemplo.

In [11]:
import pandas as pd

# Intentar diferentes codificaciones automáticamente
for encoding in ['latin-1', 'iso-8859-1', 'cp1252', 'utf-8-sig']:
    try:
        df = pd.read_csv('ventas.csv', encoding=encoding)
        print(f'✅ Archivo cargado con encoding: {encoding}')
        break
    except:
        continue

print(f'📊 {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📋 Columnas: {list(df.columns)}')
display(df.head(5))



✅ Archivo cargado con encoding: latin-1
📊 2823 filas × 25 columnas
📋 Columnas: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 📚 Celda 3: Construir el Corpus

Convertimos cada fila del CSV en un texto descriptivo que formará el corpus del RAG.

In [12]:
def fila_a_texto(row):
    """Convierte una fila del DataFrame en texto natural para el corpus."""
    return (
        f"La orden #{row['ORDERNUMBER']} del {row['ORDERDATE']} fue realizada por "
        f"'{row['CUSTOMERNAME']}' de {row['CITY']}, {row['COUNTRY']} (territorio: {row['TERRITORY']}). "
        f"Se ordenaron {row['QUANTITYORDERED']} unidades del producto {row['PRODUCTCODE']} "
        f"(línea: {row['PRODUCTLINE']}) a un precio de ${row['PRICEEACH']} cada uno, "
        f"generando una venta total de ${row['SALES']}. "
        f"Estado del pedido: {row['STATUS']}. Tamaño del trato: {row['DEALSIZE']}."
    )

# Construir corpus con todas las filas
corpus = [fila_a_texto(row) for _, row in df.iterrows()]

# Resúmenes estadísticos adicionales
resumen_general = (
    f"Resumen general: el total de ventas es ${df['SALES'].sum():,.2f}. "
    f"El cliente con mayor facturación es '{df.groupby('CUSTOMERNAME')['SALES'].sum().idxmax()}'. "
    f"La línea de producto más vendida es '{df.groupby('PRODUCTLINE')['SALES'].sum().idxmax()}'. "
    f"El país con más ventas es '{df.groupby('COUNTRY')['SALES'].sum().idxmax()}'. "
    f"El territorio con más ventas es '{df.groupby('TERRITORY')['SALES'].sum().idxmax()}'."
)

resumen_productos = (
    "Ventas por línea de producto: " +
    ". ".join([
        f"{linea}: ${total:,.2f}"
        for linea, total in df.groupby('PRODUCTLINE')['SALES'].sum().sort_values(ascending=False).items()
    ]) + "."
)

resumen_estados = (
    "Pedidos por estado: " +
    ". ".join([
        f"{estado}: {cantidad} pedidos"
        for estado, cantidad in df['STATUS'].value_counts().items()
    ]) + "."
)

resumen_dealsize = (
    "Ventas por tamaño de trato: " +
    ". ".join([
        f"{size}: ${total:,.2f}"
        for size, total in df.groupby('DEALSIZE')['SALES'].sum().items()
    ]) + "."
)

corpus.extend([resumen_general, resumen_productos, resumen_estados, resumen_dealsize])

print(f'✅ Corpus construido con {len(corpus)} fragmentos')
print(f'\nEjemplo de fragmento:\n→ {corpus[0]}')

✅ Corpus construido con 2827 fragmentos

Ejemplo de fragmento:
→ La orden #10107 del 2/24/2003 0:00 fue realizada por 'Land of Toys Inc.' de NYC, USA (territorio: nan). Se ordenaron 30 unidades del producto S10_1678 (línea: Motorcycles) a un precio de $95.7 cada uno, generando una venta total de $2871.0. Estado del pedido: Shipped. Tamaño del trato: Small.


## 🔍 Celda 4: Crear índice de vectores (FAISS)

Generamos embeddings del corpus y los indexamos con FAISS para búsqueda semántica rápida.

In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('🔄 Cargando modelo de embeddings...')
# Modelo multilingüe (soporta español)
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('🔄 Generando embeddings del corpus...')
embeddings_corpus = modelo_embeddings.encode(corpus, show_progress_bar=True)

# Crear índice FAISS
dimension = embeddings_corpus.shape[1]
indice_faiss = faiss.IndexFlatL2(dimension)
indice_faiss.add(np.array(embeddings_corpus, dtype='float32'))

print(f'\n✅ Índice FAISS creado con {indice_faiss.ntotal} vectores de dimensión {dimension}')

🔄 Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Generando embeddings del corpus...


Batches:   0%|          | 0/89 [00:00<?, ?it/s]


✅ Índice FAISS creado con 2827 vectores de dimensión 384


## 🧠 Celda 5: Configurar el modelo Mistral

> Necesitás una API Key de Mistral. Obtené una gratis en https://console.mistral.ai/

In [14]:
import requests
from google.colab import userdata

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
MODELO_MISTRAL = "mistral-small-latest"

def llamar_mistral(prompt):
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    body = {
        "model": MODELO_MISTRAL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 512,
        "temperature": 0.3
    }
    respuesta = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers=headers,
        json=body
    )
    return respuesta.json()["choices"][0]["message"]["content"]

print(f"✅ Conexión con Mistral lista | Modelo: {MODELO_MISTRAL}")

✅ Conexión con Mistral lista | Modelo: mistral-small-latest


## 🤝 Celda 6: Agente RAG — el pipeline completo

El agente combina:
1. **Recuperación**: busca los fragmentos más relevantes del corpus
2. **Aumentación**: construye un prompt con el contexto encontrado
3. **Generación**: Mistral genera la respuesta final

In [15]:
def recuperar_contexto(pregunta: str, top_k: int = 4) -> list[str]:
    """
    Agente de Recuperación: busca los fragmentos más relevantes
    del corpus usando similitud semántica.
    """
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    distancias, indices = indice_faiss.search(
        np.array(embedding_pregunta, dtype='float32'), top_k
    )
    fragmentos = [corpus[i] for i in indices[0]]
    return fragmentos


def construir_prompt(pregunta: str, fragmentos: list[str]) -> str:
    contexto = "\n".join([f"- {f}" for f in fragmentos])
    prompt = f"""Eres un asistente experto en análisis de ventas globales.
Tenés acceso a datos de ventas de múltiples países y regiones del mundo.
Respondé la pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la pregunta es sobre países o regiones, considerá TODOS los fragmentos disponibles antes de responder.
Nunca asumas que los datos son solo de un país si el contexto muestra múltiples países.
Respondé siempre en español, de forma clara, completa y con datos concretos.

CONTEXTO DE VENTAS:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""
    return prompt


def generar_respuesta(prompt: str) -> str:
    return llamar_mistral(prompt)


def agente_rag(pregunta: str, top_k: int = 4, verbose: bool = False) -> str:
    """
    Pipeline RAG completo:
    Pregunta → Recuperar → Aumentar → Generar → Respuesta
    """
    # Paso 1: Recuperar contexto relevante
    fragmentos = recuperar_contexto(pregunta, top_k=top_k)

    if verbose:
        print(f'📎 Fragmentos recuperados ({len(fragmentos)}):')
        for i, f in enumerate(fragmentos, 1):
            print(f'  {i}. {f[:100]}...')
        print()

    # Paso 2: Construir prompt con contexto
    prompt = construir_prompt(pregunta, fragmentos)

    # Paso 3: Generar respuesta con Mistral
    respuesta = generar_respuesta(prompt)

    return respuesta


print('✅ Agente RAG definido y listo')

✅ Agente RAG definido y listo


## 💬 Celda 7: Probar el bot con preguntas

In [16]:
# ─── Preguntas de prueba ───────────────────────────────────────
preguntas_prueba = [
   "¿Cuál es el cliente que más compró?"
"¿Qué línea de productos genera más ventas?"
"¿Cuántos pedidos fueron cancelados?"
"¿Qué país tiene más ventas?"
]

for pregunta in preguntas_prueba:
    print(f'\n🟡 Pregunta: {pregunta}')
    print('-' * 60)
    respuesta = agente_rag(pregunta, verbose=False)
    print(f'🟢 Respuesta: {respuesta}')
    print()


🟡 Pregunta: ¿Cuál es el cliente que más compró?¿Qué línea de productos genera más ventas?¿Cuántos pedidos fueron cancelados?¿Qué país tiene más ventas?
------------------------------------------------------------
🟢 Respuesta: Basándome **exclusivamente** en el contexto proporcionado:

1. **Cliente que más compró**:
   - **Technics Stores Inc.** (Burlingame, USA) realizó **3 pedidos** con un total de **119 unidades** (47 + 40 + 32) y ventas acumuladas de **$13,887.78** ($5,105.14 + $4,601.20 + $4,181.44).
   - **Cambridge Collectables Co.** (Cambridge, USA) realizó **1 pedido** con **32 unidades** y ventas de **$3,360.00**.
   → **Technics Stores Inc.** es el cliente con mayores ventas.

2. **Línea de productos que genera más ventas**:
   - Todos los pedidos corresponden a la línea **"Trucks and Buses"** (productos S18_1097, S24_2300, S18_4600).
   → **Trucks and Buses** es la única línea registrada en los datos, por lo que genera **el 100% de las ventas**.

3. **Pedidos cancelados**:


## 🗣️ Celda 8: Chat interactivo

Hacé preguntas libremente al bot sobre tus datos de ventas.

In [ ]:
print('🤖 Bot de Ventas RAG — Escribí "salir" para terminar')
print('=' * 60)

while True:
    pregunta = input('\n❓ Tu pregunta: ').strip()

    if pregunta.lower() in ['salir', 'exit', 'quit', 'q']:
        print('👋 ¡Hasta luego!')
        break

    if not pregunta:
        print('⚠️ Por favor ingresá una pregunta.')
        continue

    print('🔄 Buscando respuesta...')
    respuesta = agente_rag(pregunta, verbose=True)
    print(f'\n🤖 Respuesta:\n{respuesta}')

🤖 Bot de Ventas RAG — Escribí "salir" para terminar

❓ Tu pregunta: Cuantas ventas hubo?
🔄 Buscando respuesta...
📎 Fragmentos recuperados (4):
  1. La orden #10140 del 7/24/2003 0:00 fue realizada por 'Technics Stores Inc.' de Burlingame, USA (terr...
  2. La orden #10140 del 7/24/2003 0:00 fue realizada por 'Technics Stores Inc.' de Burlingame, USA (terr...
  3. La orden #10168 del 10/28/2003 0:00 fue realizada por 'Technics Stores Inc.' de Burlingame, USA (ter...
  4. La orden #10168 del 10/28/2003 0:00 fue realizada por 'Technics Stores Inc.' de Burlingame, USA (ter...


🤖 Respuesta:
Según el contexto proporcionado, hubo **4 ventas** en total. Estas corresponden a las siguientes órdenes:

1. Orden #10140 (7/24/2003): 2 ventas (productos S18_4600 y S18_1097).
2. Orden #10168 (10/28/2003): 2 ventas (productos S18_2581 y S24_1785).

Cada orden incluye múltiples productos, pero se contabilizan como ventas separadas por cada producto distinto.

❓ Tu pregunta: cual es la suma de las venta

## 📊 Celda 9 (Extra): Estadísticas del dataset

Análisis exploratorio rápido del archivo `ventas.csv`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Análisis de Ventas - Sales Data', fontsize=16, fontweight='bold')

# 1. Ventas por línea de producto
ventas_linea = df.groupby('PRODUCTLINE')['SALES'].sum().sort_values(ascending=False)
axes[0,0].bar(ventas_linea.index, ventas_linea.values, color='steelblue')
axes[0,0].set_title('Total por Línea de Producto')
axes[0,0].set_ylabel('Ventas ($)')
axes[0,0].tick_params(axis='x', rotation=30)
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# 2. Ventas por país (top 8)
ventas_pais = df.groupby('COUNTRY')['SALES'].sum().sort_values(ascending=False).head(8)
axes[0,1].barh(ventas_pais.index, ventas_pais.values, color='coral')
axes[0,1].set_title('Top 8 Países por Ventas')
axes[0,1].set_xlabel('Ventas ($)')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# 3. Distribución por tamaño de trato
dealsize = df['DEALSIZE'].value_counts()
colors = ['#4CAF50', '#2196F3', '#FF9800']
axes[0,2].pie(dealsize.values, labels=dealsize.index, autopct='%1.1f%%',
              startangle=90, colors=colors)
axes[0,2].set_title('Distribución por Deal Size')

# 4. Ventas por año
ventas_anio = df.groupby('YEAR_ID')['SALES'].sum()
axes[1,0].plot(ventas_anio.index, ventas_anio.values, marker='o',
               linewidth=2.5, color='purple', markersize=8)
axes[1,0].set_title('Ventas por Año')
axes[1,0].set_ylabel('Ventas ($)')
axes[1,0].set_xlabel('Año')
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1,0].grid(True, alpha=0.3)

# 5. Top 8 clientes
top_clientes = df.groupby('CUSTOMERNAME')['SALES'].sum().sort_values(ascending=False).head(8)
axes[1,1].barh(top_clientes.index, top_clientes.values, color='teal')
axes[1,1].set_title('Top 8 Clientes')
axes[1,1].set_xlabel('Ventas ($)')
axes[1,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# 6. Pedidos por estado
estados = df['STATUS'].value_counts()
colores_estado = ['#4CAF50','#2196F3','#FF9800','#f44336','#9C27B0','#00BCD4']
axes[1,2].bar(estados.index, estados.values, color=colores_estado[:len(estados)])
axes[1,2].set_title('Pedidos por Estado')
axes[1,2].set_ylabel('Cantidad')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Resumen en texto
print(f'\n📈 Total facturado:       ${df["SALES"].sum():>15,.2f}')
print(f'📦 Líneas de producto:    {df["PRODUCTLINE"].nunique():>15}')
print(f'👤 Clientes únicos:       {df["CUSTOMERNAME"].nunique():>15}')
print(f'🌍 Países:                {df["COUNTRY"].nunique():>15}')
print(f'🗂️  Total de órdenes:      {df["ORDERNUMBER"].nunique():>15}')
print(f'✅ Pedidos enviados:       {(df["STATUS"]=="Shipped").sum():>15}')
print(f'❌ Pedidos cancelados:     {(df["STATUS"]=="Cancelled").sum():>15}')

---

## 🗂️ Resumen del Pipeline

```
ventas.csv
    ↓
[ CORPUS ]  →  textos descriptivos de cada venta
    ↓
[ EMBEDDINGS ]  →  paraphrase-multilingual-MiniLM (FAISS)
    ↓
Pregunta del usuario
    ↓
[ AGENTE RAG ]
  1. Recuperar fragmentos relevantes (FAISS)
  2. Construir prompt con contexto
  3. Generar respuesta (Mistral)
    ↓
✅ Respuesta final
```